# 🚀 High-Performance BanglaBERT GPU Fine-Tuning (Google Colab)
### **Course:** CSE 4122: Natural Language Processing Sessional  
### **Objective:** Fine-tune 100% of all 12 transformer encoder layers on a free Nvidia T4 GPU to maximize accuracy and F1-score across Sentiment, Sarcasm, and Hate Speech.

---
### 📋 How to Run in Google Colab:
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** and click **Save**.
3. Run **Cell 1** to verify GPU.
4. In **Cell 3**, upload `cleaned_data.zip` (only ~11.8 MB).
5. Run **Cell 4** to execute GPU fine-tuning (takes ~12-15 minutes total).
6. Run **Cell 5** to download `banglabert_colab_results.zip` back to your PC.

In [ ]:
# Cell 1: Check GPU Acceleration
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    raise RuntimeError('Please enable GPU under Runtime > Change runtime type > T4 GPU!')

In [ ]:
# Cell 2: Install Modern Hugging Face Dependencies
!pip install -q transformers datasets accelerate scikit-learn seaborn matplotlib

In [ ]:
# Cell 3: Upload & Extract cleaned_data.zip
import os, zipfile
from google.colab import files

if not os.path.exists('cleaned_data'):
    print('Please upload cleaned_data.zip (~11.8 MB) from your local project...')
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('.')
    print('Extracted successfully!')

print('Available datasets:', os.listdir('cleaned_data'))

In [ ]:
# Cell 4: Full 12-Layer GPU Fine-Tuning Pipeline
import os, sys, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda')
MODEL_NAME = 'sagorsarker/bangla-bert-base'
BATCH_SIZE = 32
MAX_LEN = 96
EPOCHS = 3
LR = 2e-5

os.makedirs('saved_models', exist_ok=True)
os.makedirs('eda_plots', exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss, preds, truths = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=mask)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item() * len(labels)
            batch_preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            preds.extend(batch_preds)
            truths.extend(labels.cpu().numpy())

    acc = accuracy_score(truths, preds)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(truths, preds, average='macro', zero_division=0)
    _, _, f1_w, _ = precision_recall_fscore_support(truths, preds, average='weighted', zero_division=0)
    return total_loss / len(truths), acc, prec_m, rec_m, f1_m, f1_w, preds, truths

task_configs = {
    'sentiment': {
        'num_classes': 3,
        'class_names': ['Negative', 'Neutral', 'Positive'],
        'epochs': 2,
        'sample_train': 45000
    },
    'sarcasm': {
        'num_classes': 2,
        'class_names': ['Non-Sarcastic', 'Sarcastic'],
        'epochs': 3,
        'sample_train': None
    },
    'hate_speech': {
        'num_classes': 2,
        'class_names': ['Non-Hate', 'Hate Speech'],
        'epochs': 2,
        'sample_train': None
    }
}

all_results = {}
confusion_matrices = {}

for task_name, cfg in task_configs.items():
    print(f'\n{"="*60}\n[*] Training Task: {task_name.upper()} on GPU (All 12 Layers Unfrozen)\n{"="*60}')
    train_df = pd.read_csv(f'cleaned_data/{task_name}/train.csv')
    val_df = pd.read_csv(f'cleaned_data/{task_name}/val.csv')
    test_df = pd.read_csv(f'cleaned_data/{task_name}/test.csv')

    if cfg['sample_train'] and len(train_df) > cfg['sample_train']:
        # Balanced stratified sampling for faster convergence
        train_df = train_df.groupby('label', group_keys=False).apply(lambda x: x.sample(min(len(x), cfg['sample_train'] // cfg['num_classes']), random_state=42)).reset_index(drop=True)

    print(f'Data sizes -> Train: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}')

    counts = np.bincount(train_df['label'].values, minlength=cfg['num_classes'])
    weights = len(train_df) / (cfg['num_classes'] * np.maximum(counts, 1).astype(float))
    class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    train_loader = DataLoader(TextClassificationDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TextClassificationDataset(val_df['text'].tolist(), val_df['label'].tolist(), tokenizer), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(TextClassificationDataset(test_df['text'].tolist(), test_df['label'].tolist(), tokenizer), batch_size=BATCH_SIZE, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=cfg['num_classes']).to(device)
    # Unfreeze all 12 layers
    for param in model.parameters():
        param.requires_grad = True

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * cfg['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler()

    best_val_f1 = 0.0
    best_model_dir = f'saved_models/banglabert_{task_name}'
    os.makedirs(best_model_dir, exist_ok=True)

    for ep in range(1, cfg['epochs'] + 1):
        model.train()
        total_loss = 0.0
        t0 = time.time()
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=mask)
                loss = criterion(outputs.logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        val_loss, val_acc, _, _, val_f1, _, _, _ = evaluate_model(model, val_loader, criterion)
        print(f'Epoch {ep}/{cfg["epochs"]} [{round(time.time()-t0, 1)}s] - Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro F1: {val_f1*100:.2f}%')
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            model.save_pretrained(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)

    # Test Evaluation
    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir).to(device)
    _, test_acc, test_prec, test_rec, test_f1, test_wf1, preds, truths = evaluate_model(best_model, test_loader, criterion)
    cm = confusion_matrix(truths, preds)
    confusion_matrices[task_name] = cm.tolist()

    p_cls, r_cls, f1_cls, supp_cls = precision_recall_fscore_support(truths, preds, average=None, zero_division=0)
    class_wise = {}
    for i, cname in enumerate(cfg['class_names']):
        class_wise[cname] = {'precision': round(p_cls[i], 4), 'recall': round(r_cls[i], 4), 'f1_score': round(f1_cls[i], 4), 'support': int(supp_cls[i])}

    all_results[task_name] = {
        'task': task_name,
        'test_metrics': {
            'accuracy': round(test_acc, 4),
            'macro_precision': round(test_prec, 4),
            'macro_recall': round(test_rec, 4),
            'macro_f1': round(test_f1, 4),
            'weighted_f1': round(test_wf1, 4),
            'class_wise': class_wise
        },
        'confusion_matrix': cm.tolist()
    }
    print(f'\n[✓] {task_name.upper()} FINAL TEST RESULTS:')
    print(f'    - Accuracy:    {test_acc*100:.2f}%')
    print(f'    - Macro F1:    {test_f1*100:.2f}%')
    print(f'    - Weighted F1: {test_wf1*100:.2f}%')

with open('results_banglabert.json', 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print('\n[+] Saved results_banglabert.json successfully!')

In [ ]:
# Cell 5: Package & Download Trained Models to Your PC
import shutil
from google.colab import files

print('Zipping fine-tuned models & results...')
!zip -r -q banglabert_colab_results.zip saved_models/ results_banglabert.json

print('Triggering browser download (banglabert_colab_results.zip)...')
files.download('banglabert_colab_results.zip')
print('Done! Extract this zip into your project folder and run evaluate_master_benchmark.py.')